In [1]:
current_step = 'step_006'

In [2]:
!apt install swig cmake ffmpeg xvfb python3-opengl
!pip install pyvirtualdisplay imageio[ffmpeg]

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3-opengl is already the newest version (3.1.5+dfsg-1).
swig is already the newest version (4.0.2-1ubuntu1).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.15).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [3]:
import os

NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

%env MUJOCO_GL=egl

from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

env: MUJOCO_GL=egl


In [4]:
# Prepare to load data from google drive
from google.colab import drive
import datetime

# CONNECT TO GOOGLE DRIVE
gdrive_path = '/content/drive'
drive.mount(gdrive_path)

# DEFINE WORK DIRECTORY
workDir = os.path.join(gdrive_path, 'My Drive', 'Research', current_step)
print('WorkDir:', workDir)

log_dir = os.path.join(workDir, datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
print('LogDir:', log_dir)

# create folder if it doesn't exists
if not os.path.exists(log_dir):
  os.makedirs(log_dir)

tf_log_dir = os.path.join(workDir, 'tf_logs')
print('TfLogDir:', tf_log_dir)

# create folder if it doesn't exists
if not os.path.exists(tf_log_dir):
  os.makedirs(tf_log_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
WorkDir: /content/drive/My Drive/Research/step_006
LogDir: /content/drive/My Drive/Research/step_006/20250905-010859
TfLogDir: /content/drive/My Drive/Research/step_006/tf_logs


In [5]:
# clean content folder
import os
import shutil

# location
location = "/content"

# directories
dirs = ["sample_data", "rl-zoo", "gym_darwin_op3", "videos"]

for dir in dirs:
    path = os.path.join(location, dir)
    try:
        shutil.rmtree(path)
    except OSError as e:
        print("Error: %s : %s" % (path, e.strerror))

Error: /content/sample_data : No such file or directory
Error: /content/rl-zoo : No such file or directory
Error: /content/gym_darwin_op3 : No such file or directory
Error: /content/videos : No such file or directory


In [11]:
# Install Darwin Model
model_path = '/content/gym_darwin_op3'

if os.path.isdir(model_path):
  print(f"The directory '{model_path}' exists - git pull")
  %cd {model_path}
  !git pull
  %cd /
else:
  print(f"The directory '{model_path}' does not exist - git clone")
  !git clone --single-branch --branch {current_step} https://github.com/Gianzanti/robofei_mestrado.git {model_path}


The directory '/content/gym_darwin_op3' exists - git pull
/content/gym_darwin_op3
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (7/7), done.
Unpacking objects: 100% (11/11), 9.48 KiB | 3.16 MiB/s, done.
remote: Total 11 (delta 5), reused 7 (delta 4), pack-reused 0 (from 0)
From https://github.com/Gianzanti/robofei_mestrado
   eb34349..9ddf10b  step_006   -> origin/step_006
Updating eb34349..9ddf10b
Fast-forward
 notebooks/Research_Training.ipynb | 626 +++++++++++++++++++++++++++++++++++---
 pyproject.toml                    |   2 +-
 src/robofei/env/darwin_op3.py     |   3 +-
 3 files changed, 595 insertions(+), 36 deletions(-)
/


In [12]:
!pip install -e {model_path}

Obtaining file:///content/gym_darwin_op3
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for robofei (pyproject.toml) ... done
  Created wheel for robofei: filename=robofei-0.1.12-py3-none-any.whl size=1466 sha256=6d213547fc2ae40ed6364000bd261203799be78a393f94a33382be9682913e8a
  Stored in directory: /tmp/pip-ephem-wheel-cache-e_y0lagu/wheels/43/c3/48/682f53e738aa575222935107724428d2ce2d742b055ebb4adc
Successfully built robofei
  Attempting uninstall: robofei
    Found existing installation: robofei 0.1.11
    Uninstalling robofei-0.1.11:
      Successfully uninstalled robofei-0.1.11


In [8]:
# Install RL Zoo
trainner_path = '/content/rl-zoo'

if os.path.isdir(trainner_path):
  print(f"The directory '{trainner_path}' exists - git pull")
  %cd {trainner_path}
  !git pull
  %cd /
else:
  print(f"The directory '{trainner_path}' does not exist - git clone")
  !git clone https://github.com/Gianzanti/rl-zoo.git {trainner_path}


The directory '/content/rl-zoo' does not exist - git clone
Cloning into '/content/rl-zoo'...
remote: Enumerating objects: 3661, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 3661 (delta 37), reused 27 (delta 16), pack-reused 3597 (from 2)
Receiving objects: 100% (3661/3661), 8.53 MiB | 24.45 MiB/s, done.
Resolving deltas: 100% (2234/2234), done.


In [9]:
!pip install -e {trainner_path}

Obtaining file:///content/rl-zoo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for rl_zoo3 (pyproject.toml) ... done
  Created wheel for rl_zoo3: filename=rl_zoo3-2.7.0-0.editable-py3-none-any.whl size=4428 sha256=19f3ceb6fc9435825a8e48ee0540d772a7fb0949fa269d0af2f20f48a5bf5d6d
  Stored in directory: /tmp/pip-ephem-wheel-cache-r9w8nme2/wheels/2d/0b/3c/be4c09b7d9d2a891b5d1bc1e086a8fe4f4b4f016a0613b625c
Successfully built rl_zoo3
  Attempting uninstall: rl_zoo3
    Found existing installation: rl_zoo3 2.7.0
    Uninstalling rl_zoo3-2.7.0:
      Successfully uninstalled rl_zoo3-2.7.0


In [13]:
%cd {trainner_path}

algos_cpu = ['ppo']
algos_cuda = []
# algos_cpu = ['ppo', 'a2c']
# algos_cuda = ['ddpg', 'sac', 'td3']

n_timestep = 100_000
save_freq = min(50_000, int(n_timestep / 10))
eval_freq = min(100_000, int(n_timestep / 10))
# save_freq = 10_000
# eval_freq = 10_000

max_episode_steps = 100
wrapper = [{"gymnasium.wrappers.TimeLimit": {"max_episode_steps": max_episode_steps}}]

for algo in algos_cpu:
  print('Training:', algo)
  config = f'research_config/{algo}.yml'

  !python3 train.py --algo {algo} --env DarwinOp3-v2 -conf {config} -f "{log_dir}"\
    --tensorboard-log "{tf_log_dir}" --save-freq {save_freq} \
    --vec-env subproc --eval-freq {eval_freq} --n-eval-envs 1 --eval-episodes 10 \
    --env-kwargs keep_alive_reward:0.0 motor_max_torque:3.0 \
    ctrl_cost_weight:0.0 target_distance:0.0 forward_velocity_weight:0.0 \
    --hyperparams n_timesteps:{n_timestep} env_wrapper:"{wrapper}"\
    --device cpu
  !python -m rl_zoo3.record_video --algo {algo} --env DarwinOp3-v2 -n 2000 \
    --load-best -o "{log_dir}" -f "{log_dir}"

# for algo in algos_cuda:
#   print('Training:', algo)
#   config = f'research_config/{algo}.yml'

#   !python3 train.py --algo {algo} --env DarwinOp3-v2 -conf {config} -f "{log_dir}"\
#     --tensorboard-log "{tf_log_dir}" --save-freq {save_freq} \
#     --vec-env subproc --eval-freq {eval_freq} --n-eval-envs 1 --eval-episodes 10 \
#     --env-kwargs keep_alive_reward:0.5 motor_max_torque:3.0 \
#     ctrl_cost_weight:1e-3 target_distance:2.0 forward_velocity_weight:1.0 \
#     --hyperparams n_timesteps:{n_timestep} env_wrapper:"{wrapper}"\
#     --device cuda
#   !python -m rl_zoo3.record_video --algo {algo} --env DarwinOp3-v2 -n 2000 \
#     --load-best -o "{log_dir}" -f "{log_dir}"



/content/rl-zoo
Training: ppo
2025-09-05 01:18:09.299461: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757035089.319410   10346 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757035089.325454   10346 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1757035089.342044   10346 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1757035089.342070   10346 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1757035089.342074   10346 computation_placer.